# Checkpointing and State Recovery Tutorial

Demonstrates proper save/load patterns for btorch models, including:
- Saving model weights AND memory reset values
- Loading and resuming training
- Verifying state recovery

In [ ]:
import torch
import sys
sys.path.append('..')

from btorch.models import environ, functional
from btorch.models.init import uniform_v_

from src.model import SingleLayerGLIFRSNN
from src.checkpoint import save_checkpoint, load_checkpoint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Create and Initialize Model

In [ ]:
model = SingleLayerGLIFRSNN(
    input_dim=80,
    output_dim=10,
    n_neuron=128,
).to(device)

# Initialize
batch_size = 32
functional.init_net_state(model.rnn, batch_size=batch_size, device=device)
uniform_v_(model.neuron, set_reset_value=True)

# Store reference values
original_input_scale = model.input_scale.scale.item()
original_weight = model.input_linear.weight.clone()
original_reset_v = model.neuron.reset_v.clone() if hasattr(model.neuron, 'reset_v') else None

print(f"Original input scale: {original_input_scale:.4f}")
print(f"Original weight mean: {original_weight.mean().item():.4f}")

## 2. Simulate Training (modify weights)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Fake training step
x = torch.randn(50, batch_size, 80).to(device)

functional.reset_net(model.rnn, batch_size=batch_size)
with environ.context(dt=1.0):
    output, _ = model(x)

loss = output.pow(2).sum()
loss.backward()
optimizer.step()

# Modify scale manually (like calibration would)
model.input_scale.scale.data = torch.tensor(2.5)

trained_input_scale = model.input_scale.scale.item()
trained_weight = model.input_linear.weight.clone()

print(f"After training - input scale: {trained_input_scale:.4f}")
print(f"After training - weight mean: {trained_weight.mean().item():.4f}")

## 3. Save Checkpoint

In [ ]:
from pathlib import Path

checkpoint_path = Path("/tmp/checkpoint_test.pth")

save_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=10,
    best_acc=85.5,
    path=checkpoint_path
)

print(f"Checkpoint saved to {checkpoint_path}")

# Inspect
ckpt = torch.load(checkpoint_path, weights_only=False)
print(f"\nCheckpoint contains:")
print(f"  Epoch: {ckpt['epoch']}")
print(f"  Best acc: {ckpt['best_acc']}")
print(f"  Model keys: {len(ckpt['model_state_dict'])} entries")
print(f"  Memories RV: {list(ckpt['memories_rv'].keys())}")

## 4. Create New Model and Load

In [ ]:
model_new = SingleLayerGLIFRSNN(
    input_dim=80,
    output_dim=10,
    n_neuron=128,
).to(device)

# Must initialize state before loading!
functional.init_net_state(model_new.rnn, batch_size=batch_size, device=device)

optimizer_new = torch.optim.Adam(model_new.parameters(), lr=1e-3)

# Load
epoch, best_acc = load_checkpoint(
    path=checkpoint_path,
    model=model_new,
    optimizer=optimizer_new,
    device=device
)

print(f"Loaded from epoch {epoch}, best_acc={best_acc}")

## 5. Verify Recovery

In [ ]:
# Check weights match
loaded_weight = model_new.input_linear.weight
weight_match = torch.allclose(trained_weight, loaded_weight)

# Check scale matches
loaded_scale = model_new.input_scale.scale.item()
scale_match = abs(loaded_scale - trained_input_scale) < 1e-6

print("Recovery verification:")
print(f"  Weights match: {weight_match}")
print(f"  Scale match: {scale_match} (loaded {loaded_scale:.4f} vs saved {trained_input_scale:.4f})")

# Test forward pass works
functional.reset_net(model_new.rnn, batch_size=batch_size)
with environ.context(dt=1.0):
    output_new, _ = model_new(x)

print(f"  Forward pass: OK (output shape {output_new.shape})")

# Cleanup
checkpoint_path.unlink()
print(f"\nCheckpoint cleaned up")